# ESM2-35M secretome malleability (Reviewer R1.4)

Sequence-model analogue of the AlkSecMPNN perturbation: continue masked-LM training of **ESM2-35M** on the *same* secreted-extremophile cohort and splits, then score WT sequences with masked-marginal pseudo-log-likelihood **before vs after** fine-tuning. The pH-feature **cosine test** (score-preference direction vs the acid--base axis) is run afterwards, locally, on the same axis as the structure models.

**You upload one file:** `esm35m_colab_inputs.zip`.

**Runtime:** set Runtime → Change runtime type → **T4 GPU**.

This notebook only orchestrates the committed CLI scripts (`train_esm2_mlm.py`, `score_esm2_masked_marginals.py`) — no logic lives here.

## 1. GPU + dependencies

In [ ]:
!nvidia-smi -L
# torch ships with Colab; add the HF stack the scripts need.
!pip -q install "transformers>=4.45" "datasets>=2.20" "accelerate>=0.33" safetensors scikit-learn scipy 2>/dev/null
import transformers, torch
print("transformers", transformers.__version__, "| cuda", torch.cuda.is_available())

## 2. Upload & unzip the input bundle
Choose `esm35m_colab_inputs.zip` when prompted.

In [ ]:
import os, io, zipfile
from google.colab import files
up = files.upload()
name = [n for n in up if n.endswith('.zip')][0]
with zipfile.ZipFile(io.BytesIO(up[name])) as z:
    z.extractall('/content/esm_ft')
os.chdir('/content/esm_ft')
os.makedirs('scores', exist_ok=True); os.makedirs('runs', exist_ok=True)
print('working dir:', os.getcwd())
print('contents:', sorted(os.listdir()))
print('data:', sorted(os.listdir('data')))

## 3. Config
`SCORE_N` controls the shared scoring set. `0` = all ~10k proteins (faithful to the structure-model set, but ~6-8 h across 5 models). A pI-stratified subset (default 3000) spans the acid--base axis and finishes in ~1.5-2 h with tight bootstrap CIs. The SAME sampled proteins are scored by every model, so the before/after comparison is paired.

In [ ]:
BASE_MODEL = "facebook/esm2_t12_35M_UR50D"
EPOCHS = 10
LR = 5e-5
SCORE_N = 3000   # 0 = score all proteins
SEED = 0

ARMS = [
    ("AlkSecESM35M", "data/alkaline_case_train.csv", "data/alkaline_case_val.csv"),
    ("AcidSecESM35M", "data/acid_case_train.csv", "data/acid_case_val.csv"),
    ("NeuSecESM35M_AlkMatched", "data/alkaline_neu_train.csv", "data/alkaline_neu_val.csv"),
    ("NeuSecESM35M_AcidMatched", "data/acid_neu_train.csv", "data/acid_neu_val.csv"),
]

## 4. Build the shared scoring set (same proteins for every model)

In [ ]:
import pandas as pd, numpy as np
inp = pd.read_csv('esm_score_input.csv')
if SCORE_N and SCORE_N < len(inp):
    inp['_b'] = pd.qcut(inp['isoelectric_point'], 10, labels=False, duplicates='drop')
    per = max(1, SCORE_N // (inp['_b'].nunique()))
    inp = (inp.groupby('_b', group_keys=False)
              .apply(lambda d: d.sample(min(len(d), per), random_state=SEED))
              .drop(columns='_b').reset_index(drop=True))
inp[['Entry', 'sequence']].to_csv('score_set.csv', index=False)
print('scoring set:', len(inp), 'proteins  (pI range '
      f"{inp.isoelectric_point.min():.1f}-{inp.isoelectric_point.max():.1f})")

## 5. Score the BASE model (the 'before')

In [ ]:
!python scripts/score_esm2_masked_marginals.py \
  --model_dir {BASE_MODEL} --input_csv score_set.csv \
  --id_col Entry --seq_col sequence \
  --out_csv scores/BaseESM35M_masked_marginals.csv --model_name BaseESM35M

## 6. Continue-pretrain the 4 arms
Alkaliphile / acidophile **cases** = the steered models; the matched **neutralophile controls** are the symmetric specificity check.

In [ ]:
for name, tr, va in ARMS:
    print(f'\n===== training {name} =====')
    !python scripts/train_esm2_mlm.py \
      --train_csv {tr} --val_csv {va} --out_dir runs/{name} \
      --epochs {EPOCHS} --learning_rate {LR} --overwrite_output_dir

## 7. Score each fine-tuned model (the 'after') on the same set

In [ ]:
for name, tr, va in ARMS:
    print(f'\n===== scoring {name} =====')
    !python scripts/score_esm2_masked_marginals.py \
      --model_dir runs/{name} --input_csv score_set.csv \
      --id_col Entry --seq_col sequence \
      --out_csv scores/{name}_masked_marginals.csv --model_name {name}

## 8. Package & download
Download `esm35m_scores.zip`, unzip it into `outputs/esm35m_continual_pretraining/scores/` in the repo, then run `python paper_code/08_pca_figures/charge_pca_cosine_shift.py` — the ESM rows will appear beside the ProteinMPNN rows on the same acid--base axis, each with a bootstrap 95% CI.

In [ ]:
import shutil, glob
for f in glob.glob('runs/*/run_manifest.json'):
    shutil.copy(f, 'scores/' + f.split('/')[1] + '_run_manifest.json')
shutil.make_archive('/content/esm35m_scores', 'zip', 'scores')
from google.colab import files
files.download('/content/esm35m_scores.zip')

---
## 9. (Optional) Malleability dose curve
The 10-epoch run above barely moved ESM's charge preference — but that could be under-training. This section re-trains the alkaliphile **case** and its matched **neutralophile control** at 30/100 epochs and with the aggressive **constant-lr** recipe AlkSecMPNN used (lr 1e-4, no warmup, constant schedule), scoring each on the *same* set. If the case cosine still doesn't rotate even here, ESM's bias is genuinely entrenched; if it does, malleability exists but is slower/costlier than the structure model.

**Cost:** 8 train+score runs; scoring dominates (~15-20 min each). Trim the `SWEEP` list to shorten. Reuses `score_set.csv` from §4.

In [ ]:
SWEEP_ARMS = {
    'AlkCase': ('data/alkaline_case_train.csv', 'data/alkaline_case_val.csv'),
    'AlkNeu':  ('data/alkaline_neu_train.csv',  'data/alkaline_neu_val.csv'),
    # add 'AcidCase'/'AcidNeu' here to sweep the polar arm too
}
RECIPES = {
    'default': dict(lr=5e-5, extra=''),                                  # warmup+linear
    'const':   dict(lr=1e-4, extra='--lr_scheduler_type constant --warmup_steps 0'),
}
SWEEP = [
    ('AlkCase', 'default', 30), ('AlkCase', 'default', 100),
    ('AlkCase', 'const', 30), ('AlkCase', 'const', 100),
    ('AlkNeu', 'default', 30), ('AlkNeu', 'default', 100),
    ('AlkNeu', 'const', 30), ('AlkNeu', 'const', 100),
]

In [ ]:
import os, pandas as pd
os.makedirs('sweep_scores', exist_ok=True)
manifest = []
for arm, recipe, ep in SWEEP:
    tr, va = SWEEP_ARMS[arm]; rc = RECIPES[recipe]
    tag = f'{arm}__{recipe}__e{ep}'
    outdir = f'runs_sweep/{tag}'; scoref = f'{tag}_masked_marginals.csv'
    if os.path.exists(f'sweep_scores/{scoref}'):
        print(f'skip {tag} (already scored)'); manifest.append(dict(tag=tag, arm=arm, recipe=recipe, epochs=ep, score_file=scoref)); continue
    print(f'\n##### train {tag} #####')
    !python scripts/train_esm2_mlm.py --train_csv {tr} --val_csv {va} --out_dir {outdir} --epochs {ep} --learning_rate {rc['lr']} {rc['extra']}
    print(f'##### score {tag} #####')
    !python scripts/score_esm2_masked_marginals.py --model_dir {outdir} --input_csv score_set.csv --id_col Entry --seq_col sequence --out_csv sweep_scores/{scoref} --model_name {tag}
    manifest.append(dict(tag=tag, arm=arm, recipe=recipe, epochs=ep, score_file=scoref))
pd.DataFrame(manifest).to_csv('sweep_scores/sweep_manifest.csv', index=False)
print('\nwrote sweep_scores/sweep_manifest.csv')

## 10. Package & download the sweep
Unzip into `outputs/esm35m_continual_pretraining/sweep_scores/`, then run `python paper_code/08_pca_figures/esm_epoch_sweep.py` for the cosine-vs-epoch table + figure.

In [ ]:
import shutil
for f in glob.glob('runs_sweep/*/run_manifest.json'):
    shutil.copy(f, 'sweep_scores/' + f.split('/')[1] + '_run_manifest.json')
shutil.make_archive('/content/esm35m_sweep_scores', 'zip', 'sweep_scores')
from google.colab import files
files.download('/content/esm35m_sweep_scores.zip')